### 1: Setup & API Keys

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    raise ValueError("GOOGLE_API_KEY not found in environment variables.")

print(f"API Key loaded successfully.")

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

print("Libraries imported.")

API Key loaded successfully.


/Users/fred/Desktop/Projects/physical-trainer-rag-bot/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported.


### 2: Load and process PDFs

In [ ]:
loader = PyPDFDirectoryLoader("./training_pdfs/")
documents = loader.load()

print(f"Loaded {len(documents)} documents from PDFs.")

# Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = text_splitter.split_documents(documents)

print(f"Split documents into {len(chunks)} chunks.")

### 3: Create embeddings & vector store

In [ ]:
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"

# 1. Initialize a local embedding model
print("Loading local embedding model...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. Create the Vector Store (FAISS)
print("Creating vector database...")
vector_store = FAISS.from_documents(chunks, embeddings)

# 3. Create a retriever interface
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("Vector store created successfully using local embeddings!")

### 4: Create the strict RAG chain

In [ ]:
# Strict prompt
prompt_template = """
Answer the question using only the provided context. 
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# Initialize the LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0) # Temp=0 reduces creativity/hallucination

# Build the Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", # Stuffs all retrieved context into the prompt
    retriever=retriever,
    return_source_documents=True, # Shows which PDF page it used
    chain_type_kwargs={"prompt": prompt}
)

print("RAG System initialized.")

### 5: Test the system

In [ ]:
# Function to print the result nicely
def ask_trainer(query):
    result = qa_chain.invoke({"query": query})
    print(f"xx User Question: {query}")
    print(f"xx Trainer Answer: {result['result']}")
    print("-" * 30)
    # Print source documents to verify where it came from
    for doc in result['source_documents']:
        print(f"Source: Page {doc.metadata['page']} of {doc.metadata['source']}")

# --- Test Queries ---

# 1. Relevant question
ask_trainer("How do you perform a proper deadlift?")

# 2. Another relevant question
ask_trainer("What's the best workout for beginners?'")

# 3. A question that is DEFINITELY NOT in the PDFs
ask_trainer("Who won the 1998 World Cup?")

# 4. A question about general knowledge not in the PDFS
ask_trainer("How do I bake a chocolate cake?")

### List supported models

In [5]:
import google.generativeai as genai

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

print("Available models:")
for model in genai.list_models():
    if 'generateContent' in model.supported_generation_methods:
        print(f"- {model.name} : {model.description}")



Available models:
- models/gemini-2.5-flash : Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.
- models/gemini-2.5-pro : Stable release (June 17th, 2025) of Gemini 2.5 Pro
- models/gemini-2.0-flash-exp : Gemini 2.0 Flash Experimental
- models/gemini-2.0-flash : Gemini 2.0 Flash
- models/gemini-2.0-flash-001 : Stable version of Gemini 2.0 Flash, our fast and versatile multimodal model for scaling across diverse tasks, released in January of 2025.
- models/gemini-2.0-flash-lite-001 : Stable version of Gemini 2.0 Flash-Lite
- models/gemini-2.0-flash-lite : Gemini 2.0 Flash-Lite
- models/gemini-2.0-flash-lite-preview-02-05 : Preview release (February 5th, 2025) of Gemini 2.0 Flash-Lite
- models/gemini-2.0-flash-lite-preview : Preview release (February 5th, 2025) of Gemini 2.0 Flash-Lite
- models/gemini-exp-1206 : Experimental release (March 25th, 2025) of Gemini 2.5 Pro
- models/gemini-2.5-flash-preview-tts : 